# Aula 6 — Redes que aprendem a enxergar

**Ideia central:** Uma CNN é a Aula 2 com os kernels aprendidos em vez de escolhidos. E ela aprende o atalho mais fácil que os seus dados deixarem.

**Ementa:** Perceptron e retropropagação • Camadas convolucionais • Transfer learning • Detecção e segmentação • Aprendizado por atalho

---

In [ ]:
# !pip install torch torchvision

import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt

## 1. Preparar dados

Estrutura de pastas esperada:
```
dados/
  treino/
    sadia/
    ferrugem/
    mancha/
  val/
    sadia/
    ferrugem/
    mancha/
```

In [ ]:
# Transformações para treino (com data augmentation)
tf_treino = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Transformações para validação (sem augmentation)
tf_val = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Carregar datasets (descomente e ajuste os caminhos)
# treino = datasets.ImageFolder("dados/treino", tf_treino)
# val = datasets.ImageFolder("dados/val", tf_val)
# dl_t = torch.utils.data.DataLoader(treino, batch_size=32, shuffle=True)
# dl_v = torch.utils.data.DataLoader(val, batch_size=64)

## 2. Transfer learning com ResNet-18

Usamos pesos pré-treinados no ImageNet e ajustamos apenas a última camada.

In [ ]:
# Dispositivo (GPU se disponível)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando: {dev}")

# Carregar ResNet-18 pré-treinada
rede = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Congelar todas as camadas (extrator de features)
for p in rede.parameters():
    p.requires_grad = False

# Substituir a última camada (classificação)
num_classes = 3  # sadia, ferrugem, mancha
rede.fc = nn.Linear(rede.fc.in_features, num_classes)
rede = rede.to(dev)

print(f"Parâmetros treináveis: {sum(p.numel() for p in rede.parameters() if p.requires_grad)}")

## 3. Loop de treinamento

In [ ]:
# Otimizador (só treina a nova cabeça)
opt = torch.optim.Adam(rede.fc.parameters(), lr=1e-3)
perda = nn.CrossEntropyLoss()

# Descomente para treinar de verdade:
# for epoca in range(5):
#     rede.train()
#     for x, y in dl_t:
#         x, y = x.to(dev), y.to(dev)
#         opt.zero_grad()
#         l = perda(rede(x), y)
#         l.backward()
#         opt.step()
#     
#     # Validação
#     rede.eval()
#     certo = total = 0
#     with torch.no_grad():
#         for x, y in dl_v:
#             certo += (rede(x.to(dev)).argmax(1).cpu() == y).sum().item()
#             total += len(y)
#     print(f"Época {epoca}: acurácia de validação = {certo / total:.3f}")

print("\nCódigo pronto! Descomente e ajuste os caminhos para treinar.")

## 4. Detecção com YOLO

Para detecção de objetos (ex: plantas daninhas).

In [ ]:
# !pip install ultralytics

# from ultralytics import YOLO

# # Carregar modelo pré-treinado
# modelo = YOLO("yolov8n.pt")  # nano: rápido e leve

# # Treinar no seu dataset
# # modelo.train(data="daninhas.yaml", epochs=50, imgsz=640)

# # Validar
# # metricas = modelo.val()
# # print(f"mAP@0.5: {metricas.box.map50:.3f}")
# # print(f"mAP@0.5:0.95: {metricas.box.map:.3f}")

# # Inferência em uma imagem
# # res = modelo("talhao_drone.jpg", conf=0.25)
# # res[0].save("talhao_detectado.jpg")

print("YOLO pronto para uso! Descomente e configure o arquivo daninhas.yaml")

## 5. Aprendizado por atalho

**Cuidado:** A rede aprende qualquer pista que separe as classes, inclusive o fundo!

### Testes para detectar atalhos:

1. **Validar em dados de outra origem** (outra fazenda, câmera, safra)
2. **Mapas de saliência** (Grad-CAM): verificar se a rede olha para a lesão
3. **Mascarar o objeto**: se o modelo acerta só com o fundo, tem atalho

In [ ]:
# Exemplo de Grad-CAM (requer pytorch-grad-cam)
# !pip install pytorch-grad-cam

# from pytorch_grad_cam import GradCAM
# from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
# from pytorch_grad_cam.utils.image import show_cam_on_image

# # Camada alvo (penúltima do ResNet)
# target_layers = [rede.layer4[-1]]
# cam = GradCAM(model=rede, target_layers=target_layers)

# # Gerar mapa de ativação
# # input_tensor = ...  # sua imagem normalizada
# # grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(0)])
# # visualization = show_cam_on_image(rgb_img, grayscale_cam[0, :], use_rgb=True)

print("Grad-CAM pronto! Descomente para visualizar onde a rede olha.")

---

## Desafios

### Aquecimento
Treine a rede 3 vezes com seeds diferentes. Por que a acurácia varia? O que isso diz sobre reportar um único número?

### Mão na massa
Rode o transfer learning com fotos tiradas pela turma. Separe treino e validação por aluno fotógrafo, não por foto.

### Investigação
Aplique Grad-CAM em 20 imagens. Em quantas a região quente está na lesão? Escreva como se fosse a seção de limitações de um artigo.

In [ ]:
# Espaço para os desafios


---

## Leitura recomendada

- HAYKIN, S. *Redes neurais: princípios e prática*. 2. ed. Bookman, 2001.
- LECUN, Y. et al. Gradient-based learning applied to document recognition. IEEE, 1998.
- GEIRHOS, R. et al. Shortcut learning in deep neural networks. Nature Machine Intelligence, 2020.
- NOYAN, M. A. Uncovering bias in the PlantVillage dataset. arXiv, 2022.
- KAMILARIS, A.; PRENAFETA-BOLDÚ, F. X. Deep learning in agriculture: a survey. CEA, 2018.